# Training data prep

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.svm import SVC, LinearSVC

/home/unthinkable-lap/miniconda3/envs/kaggle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("titanic_data/train.csv", index_col="PassengerId")
data

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S


In [3]:
data.query("(Pclass == 3) and (Cabin.notna())")

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
76,0,3,"Moen, Mr. Sigurd Hansen",male,25.0,0,0,348123,7.6500,F G73,S
129,1,3,"Peter, Miss. Anna",female,NaN,1,1,2668,22.3583,F E69,C
206,0,3,"Strom, Miss. Telma Matilda",female,2.0,0,1,347054,10.4625,G6,S
252,0,3,"Strom, Mrs. Wilhelm (Elna Matilda Persson)",female,29.0,1,1,347054,10.4625,G6,S
395,1,3,"Sandstrom, Mrs. Hjalmar (Agnes Charlotta Bengt...",female,24.0,0,2,PP 9549,16.7000,G6,S
430,1,3,"Pickard, Mr. Berk (Berk Trembisky)",male,32.0,0,0,SOTON/O.Q. 392078,8.0500,E10,S
700,0,3,"Humblen, Mr. Adolf Mathias Nicolai Olsen",male,42.0,0,0,348121,7.6500,F G63,S
716,0,3,"Soholt, Mr. Peter Andreas Lauritz Andersen",male,19.0,0,0,348124,7.6500,F G73,S


In [4]:
data["Cabin"] = data["Cabin"].notna().astype("int64")

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    str    
 3   Sex       891 non-null    str    
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    str    
 8   Fare      891 non-null    float64
 9   Cabin     891 non-null    int64  
 10  Embarked  889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [6]:
# data = data.drop(columns=["Name", "Ticket", "Fare", "Embarked", "Cabin"])
# data = data.drop(columns=["Name", "Ticket"])
data = data.dropna(subset=["Embarked"])
data

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,1,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,1,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0,S
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,0,S
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,1,S
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,0,S


In [7]:
data["Embarked"].unique()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

In [8]:
data = data.replace({
    # "Sex": {"male": 0, "female": 1}
    "Embarked": {"S": 1, "C": 2, "Q": 3}
})
data["Embarked"] = data["Embarked"].astype("int64")
data

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0,1
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,1,2
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0,1
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,1,1
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0,1
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,0,1
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,1,1
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,0,1


In [9]:
data["Sex"].unique()

<StringArray>
['male', 'female']
Length: 2, dtype: str

In [10]:
data = data.replace({
    # "Sex": {"male": 0, "female": 1}
    "Sex": {"male": 1, "female": 2}
})
data

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,0,1
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",2,38.0,1,0,PC 17599,71.2833,1,2
3,1,3,"Heikkinen, Miss. Laina",2,26.0,0,0,STON/O2. 3101282,7.9250,0,1
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",2,35.0,1,0,113803,53.1000,1,1
5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,0,1
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",1,27.0,0,0,211536,13.0000,0,1
888,1,1,"Graham, Miss. Margaret Edith",2,19.0,0,0,112053,30.0000,1,1
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",2,NaN,1,2,W./C. 6607,23.4500,0,1


In [11]:
data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Name      889 non-null    str    
 3   Sex       889 non-null    object 
 4   Age       712 non-null    float64
 5   SibSp     889 non-null    int64  
 6   Parch     889 non-null    int64  
 7   Ticket    889 non-null    str    
 8   Fare      889 non-null    float64
 9   Cabin     889 non-null    int64  
 10  Embarked  889 non-null    int64  
dtypes: float64(2), int64(6), object(1), str(2)
memory usage: 83.3+ KB


In [12]:
data["Sex"] = data["Sex"].astype("int64")
# data["classist"] = data["Sex"] * data["Pclass"]

In [13]:
data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Name      889 non-null    str    
 3   Sex       889 non-null    int64  
 4   Age       712 non-null    float64
 5   SibSp     889 non-null    int64  
 6   Parch     889 non-null    int64  
 7   Ticket    889 non-null    str    
 8   Fare      889 non-null    float64
 9   Cabin     889 non-null    int64  
 10  Embarked  889 non-null    int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 83.3 KB


In [29]:
# data["age_missing"] = data["Age"].isnull().astype("int64")
# data

In [30]:
# data.groupby("Parch")[["Parch", "Survived"]].value_counts()

In [31]:
# data.groupby("SibSp")[["SibSp", "Survived"]].value_counts()

In [ ]:
# median_age = data["Age"].median()
data["Age"] = data["Age"].fillna(data["Age"].median())
data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Name      889 non-null    str    
 3   Sex       889 non-null    int64  
 4   Age       889 non-null    float64
 5   SibSp     889 non-null    int64  
 6   Parch     889 non-null    int64  
 7   Ticket    889 non-null    str    
 8   Fare      889 non-null    float64
 9   Cabin     889 non-null    int64  
 10  Embarked  889 non-null    int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 83.3 KB


In [ ]:
data["Age"] = (data["Age"] - data["Age"].mean()) / data["Age"].std()
data["Fare"] = (data["Fare"] - data["Fare"].mean()) / data["Fare"].std()

In [18]:
# test_x_vals = data[["Pclass", "Sex", "SibSp", "Parch", "Age", "age_missing"]]
# test_x_vals = data[["Pclass", "Sex", "SibSp", "Parch", "classist"]]
# test_x_vals = data[["Sex", "Pclass", "SibSp", "Parch", "Age"]]
# test_x_vals = data[["Sex", "Pclass", "SibSp", "Parch"]]
features = ["Sex", "Pclass", "SibSp", "Parch", "Age", "Fare", "Embarked", "Cabin"]
train_x_vals = data[features]
# test_x_vals = data[["Pclass", "Sex"]]
# "age_missing", 
y_vals = data["Survived"]

In [19]:
train_x_vals.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 1 to 891
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Sex       889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   SibSp     889 non-null    int64  
 3   Parch     889 non-null    int64  
 4   Age       889 non-null    float64
 5   Fare      889 non-null    float64
 6   Embarked  889 non-null    int64  
 7   Cabin     889 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 62.5 KB


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    train_x_vals,
    y_vals,
    train_size=0.7,
    shuffle=True,
    random_state=42
)

# Test data prep

In [21]:
# test_data = pd.read_csv("titanic_data/test.csv", index_col="PassengerId")
test_data = pd.read_csv("titanic_data/test.csv")
test_data

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [22]:
# cleaned_test_data = test_data[["Pclass", "Sex", "SibSp", "Parch", "Age"]]
cleaned_test_data = test_data[features]
# cleaned_test_data["age_missing"] = cleaned_test_data["Age"].isnull().astype("int64")
cleaned_test_data["Age"] = cleaned_test_data["Age"].fillna(data["Age"].median())
cleaned_test_data["Fare"] = cleaned_test_data["Fare"].fillna(cleaned_test_data["Fare"].median())

cleaned_test_data["Age"] = (cleaned_test_data["Age"] - cleaned_test_data["Age"].mean()) / cleaned_test_data["Age"].std()
cleaned_test_data["Fare"] = (cleaned_test_data["Fare"] - cleaned_test_data["Fare"].mean()) / cleaned_test_data["Fare"].std()

cleaned_test_data["Cabin"] = cleaned_test_data["Cabin"].notna().astype("int64")
cleaned_test_data = cleaned_test_data.replace({
    "Sex": {"male": 1, "female": 2}
})
cleaned_test_data = cleaned_test_data.replace({
    "Embarked": {"S": 1, "C": 2, "Q": 3}
})
cleaned_test_data["Sex"] = cleaned_test_data["Sex"].astype("int64")
cleaned_test_data["Embarked"] = cleaned_test_data["Embarked"].astype("int64")
cleaned_test_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Sex       418 non-null    int64  
 1   Pclass    418 non-null    int64  
 2   SibSp     418 non-null    int64  
 3   Parch     418 non-null    int64  
 4   Age       418 non-null    float64
 5   Fare      418 non-null    float64
 6   Embarked  418 non-null    int64  
 7   Cabin     418 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 26.3 KB


In [23]:
# data["Survived"].value_counts()

In [24]:
test_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


# Training models

#### Rf

In [25]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=42
)

In [26]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_train)
accuracy_score(y_pred=y_pred, y_true=y_train)

0.8617363344051447

In [27]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_pred=y_pred, y_true=y_test)

0.8202247191011236

In [681]:
test_data_preds = rf.predict(cleaned_test_data)
test_data_preds

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [ ]:
submission = pd.DataFrame(
    {
        "PassengerId": test_data["PassengerId"],
        "Survived": test_data_preds
    }
)
submission.to_csv("rf_submission.csv", index=False)

#### XGB

In [28]:
xg_model = XGBClassifier(
    n_estimators=5,
    max_depth=3,
    random_state=42,
    num_parallel_tree=5,
)

In [29]:
xg_model.fit(X_train, y_train)
prediction = xg_model.predict(X_train)
accuracy_score(y_pred=prediction, y_true=y_train)

0.8392282958199357

In [30]:
xg_model.fit(X_train, y_train)
prediction = xg_model.predict(X_test)
accuracy_score(y_pred=prediction, y_true=y_test)

0.8202247191011236

In [97]:
xg_test_prediction = xg_model.predict(cleaned_test_data)
xg_test_prediction

array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [98]:
submission = pd.DataFrame(
    {
        "PassengerId": test_data["PassengerId"],
        "Survived": xg_test_prediction
    }
)
submission.to_csv("xg_submission.csv", index=False)

#### SVC

In [31]:
# svc_model = SVC(
#     random_state=42,
#     kernel="linear",
#     C=1
# )
svc_model = LinearSVC(
    random_state=42,
    max_iter=20
)

svc_model.fit(X_test, y_test)
svc_predictions = svc_model.predict(X_test)
accuracy_score(y_pred=svc_predictions, y_true=y_test)

0.8277153558052435

In [ ]:
svc_predictions_test = svc_model.predict(cleaned_test_data)
submission = pd.DataFrame(
    {
        "PassengerId": test_data["PassengerId"],
        "Survived": svc_predictions_test
    }
)
submission.to_csv("svc_submission.csv", index=False)

# Hyperparameter tuning (Parameter search)

#### XGB

In [743]:
def objective(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 1, 20),
        # "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 5, 50),
        # "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        # "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        # "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        # "gamma": trial.suggest_float("gamma", 0, 5),
        "num_parallel_tree": trial.suggest_int("num_parallel_tree", 5, 50)
    }
    model = XGBClassifier(
        **param,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()
    return score

In [ ]:
study = optuna.create_study(study_name="xgboost_study", direction="maximize")
# study.optimize(objective, n_trials=1000, show_progress_bar=True, n_jobs=-1)
study.optimize(objective, n_trials=200, n_jobs=-1)

xg_best_params = study.best_params
print(f"Best params: {xg_best_params}")

[I 2026-09-29 17:55:11,515] A new study created in memory with name: xgboost_study
[W 2026-09-29 17:55:11,518] Trial 0 failed with parameters: {'max_depth': 13, 'n_estimators': 12, 'num_parallel_tree': 15} because of the following error: InvalidParameterError("The 'cv' parameter of cross_val_score must be an int in the range [2, inf), an object implementing 'split' and 'get_n_splits', an iterable or None. Got 1 instead.").
Traceback (most recent call last):
  File "/home/unthinkable-lap/miniconda3/envs/titanic/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_93599/1506344599.py", line 17, in objective
    score = cross_val_score(model, X_train, y_train, cv=1, scoring="accuracy").mean()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/unthinkable-lap/miniconda3/envs/titanic/lib/python3.12/site-packages/sklearn/utils/_param

InvalidParameterError: The 'cv' parameter of cross_val_score must be an int in the range [2, inf), an object implementing 'split' and 'get_n_splits', an iterable or None. Got 1 instead.

In [ ]:
xg_test_model = XGBClassifier(
    **xg_best_params,
    random_state=42
)

xg_test_model.fit(X_train, y_train)
prediction = xg_test_model.predict(X_train)
accuracy_score(y_pred=prediction, y_true=y_train)

0.9855305466237942

In [ ]:
prediction = xg_test_model.predict(X_test)
accuracy_score(y_pred=prediction, y_true=y_test)

0.7865168539325843

#### RF

In [ ]:
def objective(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 1, 20),
        "n_estimators": trial.suggest_int("n_estimators", 5, 100),
        # "min_samples_split": trial.suggest_int("min_samples_split", 2, 5),
        # "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
    }
    model = RandomForestClassifier(
        **param,
        random_state=42,
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()
    return score

study = optuna.create_study(study_name="random_forest_study", direction="maximize")
# study.optimize(objective, n_trials=1000, show_progress_bar=True, n_jobs=-1)
study.optimize(objective, n_trials=200, n_jobs=-1)

rf_best_params = study.best_params
print(f"Best params: {rf_best_params}")

[I 2026-09-29 17:57:39,082] A new study created in memory with name: random_forest_study
[I 2026-09-29 17:57:39,849] Trial 5 finished with value: 0.7845596432552955 and parameters: {'max_depth': 2, 'n_estimators': 17}. Best is trial 5 with value: 0.7845596432552955.
[I 2026-09-29 17:57:40,483] Trial 9 finished with value: 0.8392171435649697 and parameters: {'max_depth': 12, 'n_estimators': 28}. Best is trial 9 with value: 0.8392171435649697.
[I 2026-09-29 17:57:40,761] Trial 12 finished with value: 0.8424377554812338 and parameters: {'max_depth': 9, 'n_estimators': 35}. Best is trial 12 with value: 0.8424377554812338.
[I 2026-09-29 17:57:40,957] Trial 7 finished with value: 0.8359655642264339 and parameters: {'max_depth': 7, 'n_estimators': 46}. Best is trial 12 with value: 0.8424377554812338.
[I 2026-09-29 17:57:40,960] Trial 8 finished with value: 0.8408197076675338 and parameters: {'max_depth': 13, 'n_estimators': 42}. Best is trial 12 with value: 0.8424377554812338.
[I 2026-09-29 1

Best params: {'max_depth': 11, 'n_estimators': 59}


In [ ]:
rf_tuned = RandomForestClassifier(
    **rf_best_params,
    random_state=42
)
rf_tuned.fit(X_train, y_train)
y_pred = rf_tuned.predict(X_train)
accuracy_score(y_pred=y_pred, y_true=y_train)

0.9565916398713826

In [ ]:
y_pred = rf_tuned.predict(X_test)
accuracy_score(y_pred=y_pred, y_true=y_test)

0.8089887640449438

##### lda

In [74]:
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report


# LDA + Logistic Regression
lda_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis(n_components=1)),
    ("classifier", LogisticRegression(
        random_state=42
    ))
    # ("classifier", RandomForestClassifier(
    #     n_estimators=25,
    #     max_depth=2,
    #     random_state=42
    # ))
])




# Train
lda_model.fit(X_train, y_train)


# Predict
y_pred = lda_model.predict(X_test)


# Evaluate
accuracy = accuracy_score(y_test, y_pred)


print("LDA Accuracy:", accuracy)
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

LDA Accuracy: 0.797752808988764


In [936]:
lda_test_prediction = lda_model.predict(cleaned_test_data)
lda_test_prediction

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [937]:
submission = pd.DataFrame(
    {
        "PassengerId": test_data["PassengerId"],
        "Survived": lda_test_prediction
    }
)
submission.to_csv("lda_submission.csv", index=False)

# NN

In [79]:
train_x_vals

,Sex,Pclass,SibSp,Parch,Age,Fare,Embarked,Cabin
PassengerId,,,,,,,,
1,1,3,1,0,-5.892881e-01,-0.499958,1,0
2,2,1,1,0,6.444852e-01,0.788503,2,1
3,2,3,0,0,-2.808448e-01,-0.486376,1,0
4,2,1,1,0,4.131527e-01,0.422623,1,1
5,1,3,0,0,4.131527e-01,-0.483861,1,0
...,...,...,...,...,...,...,...,...
887,1,2,0,0,-2.037340e-01,-0.384258,1,0
888,2,1,0,0,-8.206206e-01,-0.042189,1,1
889,2,3,1,2,-5.479054e-16,-0.173986,1,0


In [145]:
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
torch.manual_seed(0)

class NeuralNetwork(torch.nn.Module):
    def __init__(self, input_units, output_units):
        super().__init__()

        # self.layers = torch.nn.Sequential(
        #     torch.nn.Linear(input_units, 256),
        #     torch.nn.ReLU(),
        #     torch.nn.Linear(256, 128),
        #     # torch.nn.ReLU(),
        #     # torch.nn.Linear(98, 28),
        #     torch.nn.ReLU(),
        #     torch.nn.Linear(128, output_units)
        # )
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(input_units, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(16, output_units)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


model = NeuralNetwork(8, 2)
# model = NeuralNetwork(4, 2)
model

numpy_feature_set = train_x_vals.to_numpy().astype(np.float32)
numpy_target_set = y_vals.to_numpy().astype(np.long)

numpy_X_train, numpy_X_test, numpy_y_train, numpy_y_test = train_test_split(
    numpy_feature_set,
    numpy_target_set,
    random_state=42,
    shuffle=True,
    train_size=0.7
)

numpy_X_train = scaler.fit_transform(numpy_X_train)
numpy_X_test = scaler.transform(numpy_X_test)


class NNDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = NNDataset(numpy_X_train, numpy_y_train)
test_ds = NNDataset(numpy_X_test, numpy_y_test)

train_loader = torch.utils.data.DataLoader(
    dataset=train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


import torch.nn.functional as F

# optimizer = torch.optim.SGD(model.parameters(), lr=0.005)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

epochs = 500

for epoch in range(epochs):
    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1:03d}/{epochs:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | Train loss: {loss:.6f}")

    model.eval()

Epoch 001/500 | Batch 000/077 | Train loss: 0.712288
Epoch 001/500 | Batch 001/077 | Train loss: 0.661511
Epoch 001/500 | Batch 002/077 | Train loss: 0.700225
Epoch 001/500 | Batch 003/077 | Train loss: 0.680272
Epoch 001/500 | Batch 004/077 | Train loss: 0.735258
Epoch 001/500 | Batch 005/077 | Train loss: 0.631393
Epoch 001/500 | Batch 006/077 | Train loss: 0.642724
Epoch 001/500 | Batch 007/077 | Train loss: 0.742257
Epoch 001/500 | Batch 008/077 | Train loss: 0.583033
Epoch 001/500 | Batch 009/077 | Train loss: 0.624915
Epoch 001/500 | Batch 010/077 | Train loss: 0.626005
Epoch 001/500 | Batch 011/077 | Train loss: 0.677602
Epoch 001/500 | Batch 012/077 | Train loss: 0.768891
Epoch 001/500 | Batch 013/077 | Train loss: 0.548415
Epoch 001/500 | Batch 014/077 | Train loss: 0.583127
Epoch 001/500 | Batch 015/077 | Train loss: 0.630203
Epoch 001/500 | Batch 016/077 | Train loss: 0.567842
Epoch 001/500 | Batch 017/077 | Train loss: 0.538804
Epoch 001/500 | Batch 018/077 | Train loss: 0.

In [146]:
with torch.no_grad():
    # out = torch.argmax(model(torch.from_numpy(numpy_X_test)), dim=1)
    logits = model(torch.from_numpy(numpy_X_train))
    out = torch.argmax(logits, dim=1)

accuracy_score(y_pred=out.detach().cpu().numpy(), y_true=numpy_y_train)

0.887459807073955

In [147]:
with torch.no_grad():
    # out = torch.argmax(model(torch.from_numpy(numpy_X_test)), dim=1)
    logits = model(torch.from_numpy(numpy_X_test))
    out = torch.argmax(logits, dim=1)

accuracy_score(y_pred=out.detach().cpu().numpy(), y_true=numpy_y_test)

0.8052434456928839